In [ ]:
import uclchem
import numpy as np

# Class: GridRunner

The GridRunner class, allows users to provide the input values for a given chemical model class (i.e. Cloud, PrestellarCore, JShock, etc.), containing singular and/or list/array values for parameters. The class will create a grid of all possible parameter combinations and then run a model of the selected chemical model class for each combination. The resulting models are run in parallel and stored in the given path provided to grid_file.

## Was this possible before?

Yes, it was and is possible to do this without the GridRunner class. It requires creating the parameter grid externally to UCLCHEM, as well as understanding how to set up the parallel model calculations. This can be done in many different ways, but since it is becoming a standard way of executing UCLCHEM models, we have added the ability to do this internally to UCLCHEM.

In [ ]:
ParameterDictionary = {
    "param_dict": {
        "endatfinaldensity": False,
        "freefall": False,
        "initialDens": np.logspace(2, 4, 3), # This creates an array containing the values [100, 1000, 10000],
        "initialTemp": np.linspace(10, 15, 6), # This creates an array containing the values [10, 11, 12, 13, 14, 15]
        "finalDens": 1e4,
        "finalTime": 1.0e5,
    }
}

grid = uclchem.model.GridRunner(
    model_type="Cloud",
    full_parameters=ParameterDictionary,
    max_workers=9,
    grid_file="./grid_basic.h5",
    model_name_prefix="basic_grid_"
)

## Where are my models?

GridRunner runs each model in the defined grid, and saves the models to the file given in the grid_file input. Once it saves a model, it removes the model from the RAM, in order to free up space and prevent crashing a system by overusing RAM.

To help you know which models are part of your grid, the changing parameter values, as well as the model names can be found in the ```.models``` attribute. The name of a model is stored in the key 'Model'. The prefix of the name will always be the given model_name_prefix, followed by the number of the model in the grid.

In [ ]:
grid.models

So to access the fifth model, we do:

In [ ]:
model_5 = uclchem.model.load_model(
    file = "./grid_basic.h5",
    name = "basic_grid_5"
)

And from here, we can used the model in the same was as all other chemical models.

In [ ]:
model_5.check_conservation()

# Class: SequentialRunner

The SequentialRunner class, allows users to predefine a specific order of models to run, such that previous models pass their chemical abundances to the next model. This is done so that the next model can use this chemistry as a starting chemistry.

## By Hand Sequencing

By now, you have seen how using the final chemical abundances of a model are used as the starting abundances for another model, but just to give a clear comparison, here is how one can run a sequential model without using the SequentialRunner class.

In [ ]:
cloud_param_dict = {
    "endAtFinalDensity": False,  # stop at finalTime
    "freefall": True,  # increase density in freefall
    "initialDens": 1e2,  # starting density
    "finalDens": 1e6,  # final density
    "initialTemp": 10.0,  # temperature of gas
    "finalTime": 6.0e6,  # final time
    "rout": 0.1,  # radius of cloud in pc
    "baseAv": 1.0,  # visual extinction at cloud edge.
}
cloud = uclchem.model.Cloud(param_dict=cloud_param_dict)

In [ ]:
prestellar_core_param_dict = {
    **cloud_param_dict, # This copies all keys and values from the dictionary.
    "initialDens": 1e6, # We do this, in this case, to match the intended finaldensity from the previous model with the starting density of this model.
    "finalTime": 1e5,
    "freefall": False,
    # freeze out is completely overwhelmed by thermal desorption
    # so turning it off has no effect on abundances but speeds up integrator.
    "freezeFactor": 0.0,
    # Set the tolerances we need to solve the model effectively:
    "abstol_factor": 1e-18,
    "reltol": 1e-12
}
# Starting with **cloud_param_dict makes it so new keys overwrite pre-existing keys copied from cloud_param_dict.

# We pass cloud as the previous_model so that PrestellarCore can access the final chemical abundances to set the starting chemical abundances.
p_core = uclchem.model.PrestellarCore(
    temp_indx=3, max_temperature=300.0, param_dict=prestellar_core_param_dict, previous_model=cloud
)

p_core.check_conservation()

## Using SequentialRunner

In using the SequentialRunner class, we have to define the parameters of each model along the sequence. Doing so, we can explicitly

In [ ]:
config = [
    {
        "Cloud": {
            "param_dict": {
                "endAtFinalDensity": False,  # stop at finalTime
                "freefall": True,  # increase density in freefall
                "initialDens": 1e2,  # starting density
                "finalDens": 1e6,  # final density
                "initialTemp": 10.0,  # temperature of gas
                "finalTime": 6.0e6,  # final time
                "rout": 0.1,  # radius of cloud in pc
                "baseAv": 1.0,  # visual extinction at cloud edge.
            },
        },
    },
    {
        "PrestellarCore": {
            "param_dict": {
                "endAtFinalDensity": False,  # stop at finalTime
                "freefall": False,  # increase density in freefall
                "initialTemp": 10.0,  # temperature of gas
                "finalTime": 1e5,  # final time
                "rout": 0.1,  # radius of cloud in pc
                "baseAv": 1.0,  # visual extinction at cloud edge.
                # freeze out is completely overwhelmed by thermal desorption
                # so turning it off has no effect on abundances but speeds up integrator.
                "freezeFactor": 0.0,
                # Set the tolerances we need to solve the model effectively:
                "abstol_factor": 1e-18,
                "reltol": 1e-12,
            },
            "temp_indx": 3,
            "max_temperature": 300.0
        },
    },
]

seq_model = uclchem.model.SequentialRunner(config, parameters_to_match=["finalDens"])

To access a specific model in the sequence, we can access the list of models, the attribute 'SequentialRunner.models'. The model object, the one with the attributes and methods we can use, is located in the key 'Model' within the dictionary in the element of interest.

So, to access the first model in the sequence, we would access ```seq_model.models[0]['Model']```, to get the last one we can do ```seq_model.models[-1]['Model']```

In [ ]:
seq_model.models[-1]['Model'].get_dataframes()

# But what if I need to run a grid of sequential models?!

Then you are in luck. You can be our guinea pig! The GridRunner class, is being tested for compatible with the SequentialRunner class as model_type input.

We provide the parameters_to_match=["finalDens"] input to the SequentialRunner class, so that the density of the final timestep from the previous model, is explicitly used as the starting point of the next. In cases where this is the intended behaviour, providing this input is recommended, as it is possible for a model to finish prior to reaching the intended final density, when endAtFinalDensity is False.

In this instance, the SequentialRunner class approach may seem longer because we explicitly redeclare all the parameters in the param_dict. This is intentional to prevent accidentally leaving parameters in a param_dict that were not meant to be used for subsequent models along the sequence. This can be explicitly done for the traditional approach as well. For this example we used the example notebook format, as we suspect people may follow a similar way of coding, despite the potential risk of leaving parameters in the param_dict that weren't meant to remain.

In [ ]:
sequenced_model_parameters_dict = [
    {
        "Cloud": {
            "param_dict": {
                "endAtFinalDensity": False,  # stop at finalTime
                "freefall": True,  # increase density in freefall
                "initialDens": np.logspace(2, 3, 2),  # starting density
                "finalDens": 1e6,  # final density
                "initialTemp": 10.0,  # temperature of gas
                "finalTime": 6.0e6,  # final time
                "rout": 0.1,  # radius of cloud in pc
                "baseAv": 1.0,  # visual extinction at cloud edge.
            }
        },
    },
    {
        "PrestellarCore": {
            "param_dict": {
                "endAtFinalDensity": False,  # stop at finalTime
                "freefall": False,  # increase density in freefall
                "initialTemp": 10.0,  # temperature of gas
                "finalTime": 1e5,  # final time
                "rout": np.linspace(0.1, 0.2, 2),  # radius of cloud in pc
                "baseAv": 1.0,  # visual extinction at cloud edge.
                "freezeFactor": 0.0,
                "abstol_factor": 1e-18,
                "reltol": 1e-12,

            },
            # PrestellarCore has additional inputs outside param_dict that need to be filled and can be part of the grid.
            "temp_indx": 3,
            "max_temperature": [300.0, 400.0]
        },
    },
    {
        # Because the Sequential class is capable of matching final temperature and density, we can provide those inputs to the SequentialRunner here.
        "parameters_to_match": ["finalDens", "finalTemp"]
    }
]

grid = uclchem.model.GridRunner(
    model_type="SequentialRunner",
    full_parameters=sequenced_model_parameters_dict,
    max_workers=9,
    grid_file="./grid_sequence.h5",
    model_name_prefix="seq_grid_"
)

In [ ]:
grid.models

Like in the previous grid model, we can access the individual model by loading them. This time, we have to take into consideration the order of the sequential models and the model type as well. So we look for the model that starts with the name prefix, followed by the grid model number, and then add '_<sequence number>_<model type>' So, the fifth grid models second sequence model in our example, would have the name ```basic_grid_5_1_PrestellarCore```.

In [ ]:
n = 0
model_n_second_sequence = uclchem.model.load_model(
    file = "./grid_sequence.h5",
    name = f"seq_grid_{n}_1_PrestellarCore"
)

In [ ]:
model_n_second_sequence.check_conservation()

In [ ]:
model_n_second_sequence.get_dataframes()

From here, try seeing how you can adapt these runners into the previous examples covered in the workshop.